# Task Decomposition Workflow: Practice Exercise

Build a document analysis workflow using LangGraph that routes between single-document summarization and multi-document comparison based on input analysis.

**What you'll implement:**
- A document analyzer node that determines the processing strategy
- A router function for conditional edge routing
- The complete graph assembly with conditional edges

**Estimated time:** 15-20 minutes

## Setup

Run the cell below to import dependencies and initialize the LLM.

In [ ]:
# Setup - run this cell first

from typing import TypedDict, List, Literal
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os

load_dotenv()

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Setup complete!")

## Your Task

You will build a **document analysis workflow** that:

1. Analyzes the input to determine if it contains a single document or multiple documents
2. Routes to the appropriate processing node:
   - **Summarize**: For single documents - creates a concise summary
   - **Compare**: For multiple documents - identifies similarities and differences
3. Returns the final analysis result

**Workflow structure:**
```
START --> document_analyzer --> [conditional routing]
                                 |--> summarize --> END
                                 |--> compare --> END
```

**Input:** A dictionary with:
- `"documents"`: A list of document strings (1 or more)

**Output:** The state dictionary with `"result"` populated containing the analysis.

## Provided Components

The workflow state and processing nodes are provided for you. Review them to understand the structure.

In [ ]:
# Workflow State - provided

class DocumentState(TypedDict):
    """State schema for document analysis workflow."""
    documents: List[str]          # Input documents to process
    processing_strategy: str       # 'summarize' or 'compare'
    result: str                    # Final analysis result

print("State schema defined!")
print("\nState fields:")
for field, field_type in DocumentState.__annotations__.items():
    print(f"  - {field}: {field_type}")

In [ ]:
# Pydantic schema for structured output - provided

class DocumentAnalysis(BaseModel):
    """Schema for document analysis results."""
    
    processing_strategy: Literal["summarize", "compare"] = Field(
        description="Processing strategy: 'summarize' for single document, 'compare' for multiple documents"
    )
    reasoning: str = Field(
        description="Brief explanation of why this strategy was chosen"
    )

# Create structured output analyzer
document_analyzer_llm = llm.with_structured_output(DocumentAnalysis)

print("Document analyzer schema ready!")

In [ ]:
# Processing nodes - provided

def summarize_node(state: DocumentState) -> DocumentState:
    """
    Summarizes a single document.
    """
    document = state["documents"][0]
    
    print("\n" + "="*60)
    print("SUMMARIZE NODE")
    print("="*60)
    print(f"Processing single document ({len(document)} chars)...")
    
    prompt = f"""Provide a concise summary of the following document:

{document}

Summary:"""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    
    print("Summary generated!")
    
    return {
        **state,
        "result": response.content
    }


def compare_node(state: DocumentState) -> DocumentState:
    """
    Compares multiple documents to find similarities and differences.
    """
    documents = state["documents"]
    
    print("\n" + "="*60)
    print("COMPARE NODE")
    print("="*60)
    print(f"Comparing {len(documents)} documents...")
    
    docs_text = "\n\n---\n\n".join(
        [f"Document {i+1}:\n{doc}" for i, doc in enumerate(documents)]
    )
    
    prompt = f"""Compare the following documents and identify key similarities and differences:

{docs_text}

Provide your analysis with:
1. Key similarities
2. Key differences
3. Overall conclusion"""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    
    print("Comparison complete!")
    
    return {
        **state,
        "result": response.content
    }

print("Processing nodes defined!")

## Implementation

Complete the three components below to build the workflow.

### Part 1: Document Analyzer Node

Implement the analyzer node that examines the input documents and determines the processing strategy.

In [ ]:
def document_analyzer_node(state: DocumentState) -> DocumentState:
    """
    Analyzes input documents and determines processing strategy.
    
    This node should:
    1. Get the documents list from state
    2. Create a prompt that describes the documents (count, brief preview)
    3. Use document_analyzer_llm to get structured output
    4. Return updated state with 'processing_strategy' set
    
    Args:
        state: Current workflow state with 'documents' field populated
    
    Returns:
        Updated state with 'processing_strategy' set to 'summarize' or 'compare'
    """
    print("\n" + "="*60)
    print("DOCUMENT ANALYZER NODE")
    print("="*60)
    
    # TODO: Implement the document analyzer logic
    #
    # Steps:
    # 1. Get documents from state["documents"]
    # 2. Print how many documents were received
    # 3. Create a system prompt explaining the task (determine if summarize or compare)
    # 4. Create a user prompt with document count and brief previews (first 100 chars each)
    # 5. Call document_analyzer_llm.invoke() with the messages
    # 6. Print the chosen strategy and reasoning
    # 7. Return state with processing_strategy updated
    
    pass

### Part 2: Router Function

Implement the router function that returns the next node name based on the processing strategy.

In [ ]:
def route_by_strategy(state: DocumentState) -> Literal["summarize", "compare"]:
    """
    Routes to the appropriate processing node based on strategy.
    
    This function is used with add_conditional_edges() to create
    branching logic in the workflow graph.
    
    Args:
        state: Current workflow state with 'processing_strategy' field
    
    Returns:
        The name of the next node: 'summarize' or 'compare'
    """
    # TODO: Implement the router
    #
    # Steps:
    # 1. Get the processing_strategy from state
    # 2. Print which node is being routed to
    # 3. Return the strategy string
    
    pass

### Part 3: Build the Graph

Assemble the complete workflow graph with nodes and edges.

In [ ]:
def build_document_workflow():
    """
    Builds and compiles the document analysis workflow graph.
    
    The workflow should have:
    - 3 nodes: document_analyzer, summarize, compare
    - Entry edge: START -> document_analyzer
    - Conditional edges: document_analyzer -> summarize OR compare
    - Exit edges: summarize -> END, compare -> END
    
    Returns:
        Compiled LangGraph application
    """
    # TODO: Build and compile the graph
    #
    # Steps:
    # 1. Create StateGraph with DocumentState
    # 2. Add nodes: "document_analyzer", "summarize", "compare"
    # 3. Add edge from START to "document_analyzer"
    # 4. Add conditional edges from "document_analyzer" using route_by_strategy
    #    Map {"summarize": "summarize", "compare": "compare"}
    # 5. Add edges from "summarize" and "compare" to END
    # 6. Compile and return the graph
    
    pass


# Build the workflow
app = build_document_workflow()
print("Workflow built successfully!")

## Run Your Implementation

Test your workflow with single and multiple documents.

In [ ]:
# Test 1: Single document (should route to summarize)
single_doc = {
    "documents": [
        """Artificial intelligence has transformed many industries in recent years. 
        Machine learning models can now analyze vast amounts of data to identify patterns 
        and make predictions. Companies are using AI for customer service chatbots, 
        fraud detection, medical diagnosis, and autonomous vehicles. However, there are 
        concerns about job displacement and the need for ethical AI development."""
    ],
    "processing_strategy": "",
    "result": ""
}

print("#" * 60)
print("TEST 1: Single Document")
print("#" * 60)

result_1 = app.invoke(single_doc)

print("\n" + "="*60)
print("FINAL RESULT")
print("="*60)
print(f"Strategy: {result_1['processing_strategy']}")
print(f"\nResult:\n{result_1['result']}")

In [ ]:
# Test 2: Multiple documents (should route to compare)
multi_docs = {
    "documents": [
        """Python is a versatile programming language known for its readable syntax. 
        It excels in data science, web development, and scripting. Python has a large 
        ecosystem of libraries like NumPy, Pandas, and Django.""",
        
        """JavaScript is the language of the web, running in browsers and on servers 
        via Node.js. It powers interactive websites and modern web applications. 
        Popular frameworks include React, Vue, and Angular."""
    ],
    "processing_strategy": "",
    "result": ""
}

print("\n" + "#" * 60)
print("TEST 2: Multiple Documents")
print("#" * 60)

result_2 = app.invoke(multi_docs)

print("\n" + "="*60)
print("FINAL RESULT")
print("="*60)
print(f"Strategy: {result_2['processing_strategy']}")
print(f"\nResult:\n{result_2['result']}")